# ML Assignment 2: Breast Cancer Classification
Run this notebook in the BITS Virtual Lab. Dataset: UCI Breast Cancer Wisconsin (Diagnostic), provided through scikit-learn.

**Target meaning:** 0 = benign, 1 = malignant. **This is an educational exercise, not a medical tool.**

In [8]:
# Import the Python libraries we need.
import pandas as pd
import joblib
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix


In [9]:
# Load the dataset from scikit-learn (originally from UCI).
data = load_breast_cancer(as_frame=True)
X = data.data                  # X contains the 30 input features.
y = (data.target == 0).astype(int)  # Change target: 1 = malignant, 0 = benign.
print("Number of rows:", len(X))
print("Number of features:", X.shape[1])
print("Missing values:", X.isnull().sum().sum())
print(X.head())


Number of rows: 569
Number of features: 30
Missing values: 0
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ... 

In [10]:
# Split into 80% training and 20% test data.
# stratify=y keeps the class proportions similar in both parts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Save the test data with the target column for the Streamlit upload.
test_data = X_test.copy()
test_data["target"] = y_test
test_data.to_csv("test_data.csv", index=False)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 455
Testing rows: 114


In [11]:
# Create all five models.
# Scaling is used for Logistic Regression, KNN and SVM.
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))
}

# The saved file name for each model.
file_names = {
    "Logistic Regression": "logistic.joblib",
    "Decision Tree": "tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}


In [12]:
# Make a folder to save trained models.
Path("model").mkdir(exist_ok=True)
joblib.dump(list(X.columns), "model/feature_names.joblib")

# This list will store the scores for our comparison table.
results = []

for name, model in models.items():
    model.fit(X_train, y_train)  # Train only using training data.
    predictions = model.predict(X_test)  # Predict on unseen test data.
    probabilities = model.predict_proba(X_test)[:, 1]  # Chance of malignant class.

    # Calculate the six metrics required by the assignment.
    accuracy = accuracy_score(y_test, predictions)
    auc = roc_auc_score(y_test, probabilities)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    mcc = matthews_corrcoef(y_test, predictions)

    results.append([name, accuracy, auc, precision, recall, f1, mcc])
    joblib.dump(model, "model/" + file_names[name])  # Save the trained model.
    print("\n", name)
    print("Confusion matrix:\n", confusion_matrix(y_test, predictions))

# Display the results in one table.
columns = ["ML Model Name", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
comparison = pd.DataFrame(results, columns=columns)
print(comparison.round(4).to_string(index=False))
comparison.to_csv("model_comparison.csv", index=False)



 Logistic Regression
Confusion matrix:
 [[71  1]
 [ 3 39]]

 Decision Tree
Confusion matrix:
 [[70  2]
 [ 7 35]]

 KNN
Confusion matrix:
 [[71  1]
 [ 4 38]]

 Naive Bayes
Confusion matrix:
 [[72  0]
 [ 7 35]]

 Support Vector Machine
Confusion matrix:
 [[72  0]
 [ 3 39]]
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9649 0.9960     0.9750  0.9286 0.9512 0.9245
         Decision Tree    0.9211 0.9448     0.9459  0.8333 0.8861 0.8299
                   KNN    0.9561 0.9823     0.9744  0.9048 0.9383 0.9058
           Naive Bayes    0.9386 0.9934     1.0000  0.8333 0.9091 0.8715
Support Vector Machine    0.9737 0.9947     1.0000  0.9286 0.9630 0.9442


C:\Users\dines\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [13]:
# Select a winner using the highest F1 score.
# You can also discuss AUC, MCC and recall in your observations.
best_row = comparison.loc[comparison["F1"].idxmax()]
print("Highest F1 model:", best_row["ML Model Name"])
print("F1 score:", round(best_row["F1"], 4))


Highest F1 model: Support Vector Machine
F1 score: 0.963
